In [1]:
from google.cloud import bigquery
from google.oauth2 import service_account

PROJECT_ID = "whiteberry-network"

credentials = service_account.Credentials.from_service_account_file(
    r"..\..\credentials\service-account.json"
)

client = bigquery.Client(
    project=PROJECT_ID,
    credentials=credentials
)

#### Número de clientes: 600

In [ ]:
df = client.query("SELECT COUNT(*) FROM `whiteberry-network.whiteberry.customers`").to_dataframe()
print(df)


C:\Users\m_fon\AppData\Roaming\Python\Python312\site-packages\google\cloud\bigquery\table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


   f0_
0  600


#### Número de productos: 100

In [3]:
df = client.query("SELECT COUNT(*) FROM `whiteberry-network.whiteberry.products`").to_dataframe()
print(df)

   f0_
0  100


C:\Users\m_fon\AppData\Roaming\Python\Python312\site-packages\google\cloud\bigquery\table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


#### Número de pedidos: 2000

In [4]:
df = client.query("SELECT COUNT(*) FROM `whiteberry-network.whiteberry.orders`").to_dataframe()
print(df)

C:\Users\m_fon\AppData\Roaming\Python\Python312\site-packages\google\cloud\bigquery\table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


    f0_
0  2000


#### Líneas de pedido: 4500

In [5]:
df = client.query("SELECT COUNT(*) FROM `whiteberry-network.whiteberry.order_items`").to_dataframe()
print(df)

    f0_
0  4500


C:\Users\m_fon\AppData\Roaming\Python\Python312\site-packages\google\cloud\bigquery\table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


#### Pagos correspondientes a los pedidos: 2000

In [6]:
df = client.query("SELECT * FROM `whiteberry-network.whiteberry.payments`").to_dataframe()
print(df)

C:\Users\m_fon\AppData\Roaming\Python\Python312\site-packages\google\cloud\bigquery\table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


      payment_id  order_id payment_date payment_method payment_type    amount  \
0              4         4   2024-12-05  bank_transfer      payment    356.73   
1              6         6   2024-08-03  bank_transfer      payment   9902.84   
2             14        14   2025-12-24  bank_transfer      payment  10267.24   
3             19        19   2024-05-28  bank_transfer      payment   9221.70   
4             28        28   2026-03-01  bank_transfer      payment   7537.96   
...          ...       ...          ...            ...          ...       ...   
1995        1962      1962   2025-05-27         paypal       refund   4637.44   
1996        1968      1968   2026-01-23         paypal       refund   7078.52   
1997        1978      1978   2026-01-02         paypal       refund  19893.88   
1998        1980      1980   2024-05-15         paypal       refund   5084.58   
1999        1989      1989   2025-12-27         paypal       refund    432.03   

         status  
0     com

#### Valoraciones para ~35% de los productos entregados

Contamos:
- El número de líneas de pedido con review  
- Líneas de pedido entregadas (subtabla "d" de líneas correspondientes a pedidos entregados)
- Porcentaje que representan (diviendo entre el total de líneas de pedido y multiplicando por 100)

In [ ]:
query = """ 
SELECT
    COUNT(DISTINCT r.order_item_id) AS items_con_review,
    COUNT(DISTINCT d.order_item_id) AS items_entregados,
    ROUND((COUNT(DISTINCT r.order_item_id) / COUNT(DISTINCT d.order_item_id)) * 100, 2)AS porcentaje
FROM 
    (
        SELECT 
            oi.order_item_id,
            oi.product_id
        FROM `whiteberry-network.whiteberry.orders` o
        JOIN `whiteberry-network.whiteberry.order_items` oi
            ON o.order_id = oi.order_id
        WHERE o.status = 'delivered'
    ) AS d
LEFT JOIN 
    (
        SELECT 
            order_item_id
        FROM `whiteberry-network.whiteberry.reviews`
    ) AS r
ON d.order_item_id = r.order_item_id"""

df = client.query(query).to_dataframe()
print(df.head())

   items_con_review  items_entregados  porcentaje
0               604              1726       34.99


C:\Users\m_fon\AppData\Roaming\Python\Python312\site-packages\google\cloud\bigquery\table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(
